In [ ]:
%pip install transformers

In [ ]:
# Getting Image

from google.colab import files
from PIL import Image

uploaded_file = files.upload()
img_name = list(uploaded_file.keys())[0]

img = Image.open(img_name).convert("RGB")

img

In [ ]:
# Downloading TrOCR Model

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

trocr_model_name = "microsoft/trocr-base-printed"

img_processor = TrOCRProcessor.from_pretrained(trocr_model_name)

trocr_model = VisionEncoderDecoderModel.from_pretrained(trocr_model_name)

In [ ]:
# Converting Raw image to Tokens Vectors

img_tokens = img_processor(img, return_tensors="pt")

# Processing Token Vectors to Understand The Texts in The Image By TrOCR Model

img_value = trocr_model.generate(**img_tokens)

# Decode the Processed Token from Model to Human Readble Form

img_text = img_processor.batch_decode(img_value[0],skip_special_tokens=True)

img_text

In [ ]:
extracted_text = img_text[0]

extracted_text              # will be used later to Collect information About It

In [ ]:
# Preparing To get Qwen Model

from transformers import AutoTokenizer, AutoModelForCausalLM

model = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model)

llm_model = AutoModelForCausalLM.from_pretrained(model, torch_dtype='auto')

In [ ]:
# Making Internet Search Tool with Exa

%pip install langchain-exa

from langchain_exa import ExaSearchResults

search_tool = ExaSearchResults(
    exa_api_key="API_KEY",
    max_results=1
)

In [ ]:
# Making Instructions message for Qwen Model to follow

message_for_query = [
    {'role':'system',
     'content':""" you look at input,
      use search tool as many time as you need, sometimes you will need many times,
      to search about it."""},
    {'role': 'user',
     'content':extracted_text}
]

# Declare Tool for Qwen Model to use

TOOLS = [
    {
    'type':'function',
    'function':{
        'name':'search_tool',
        'description':"Search from Internet",
        'parameters':{
            'type':'object',
            'properties':{
                'query':{'type':'string','description':"Search about query"}
            },
            'required':['query']
        }
    }
}]

# Making Chat Template, let the qwen model to read

text_for_model = tokenizer.apply_chat_template(
    message_for_query,
    tools=TOOLS,
    tokenize=False,
)

In [ ]:
# Transform and Generate

tokens_for_model = tokenizer(text_for_model, return_tensors='pt')

output_tokens = llm_model.generate(**tokens_for_model, max_new_tokens=400)

new_tokens = output_tokens[0][tokens_for_model['input_ids'].shape[1]:]

output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(output_text)

In [ ]:
import re

texts_extracted_llm = re.findall(r"<tool_call>\s*(.*)\s*</tool_call>", output_text)

print(texts_extracted_llm)

In [ ]:
# Extract Query part to search in internet

import json

pure_extracted_text = [
    json.loads(i) for i in texts_extracted_llm if json.loads(i)['name']=="search_tool"
]

pure_extracted_text

In [ ]:
import re

def clean_text(text):
    """Remove Markdown tables, links, references, and formatting."""

    # Remove all Markdown table rows
    text = re.sub(
        r'^\s*\|.*\|\s*$',
        '',
        text,
        flags=re.MULTILINE
    )

    # Remove Markdown links but keep link text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove reference markers: [1], [2], etc.
    text = re.sub(r'\[\d+\]', '', text)

    # Remove Markdown formatting
    text = re.sub(r'[#*_`]', '', text)

    # Remove any remaining pipe characters
    text = text.replace('|', '')

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def remove_unnecessary_sections(text):
    """Remove sections that are usually not useful for answering queries."""

    # Remove long synonym sections
    text = re.sub(
        r'\bSynonyms?\s*:.*?(?=\b(?:References|External links|See also)\b|$)',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Remove external links and references
    text = re.sub(
        r'\b(?:References|External links)\b.*$',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return text.strip()


def split_sentences(text):
    """Split text into individual sentences."""

    return re.split(r'(?<=[.!?])\s+', text)


def extract_useful_text(text, query, max_sentences=10):
    text = clean_text(text)
    text = remove_unnecessary_sections(text)

    sentences = split_sentences(text)

    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    useful_sentences = []

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        sentence_words = set(
            re.findall(r'\b\w+\b', sentence.lower())
        )

        # Keep sentences containing at least one query word
        if query_words.intersection(sentence_words):
            useful_sentences.append(sentence)

    # Remove duplicate sentences
    useful_sentences = list(dict.fromkeys(useful_sentences))

    return ' '.join(useful_sentences[:max_sentences])



In [ ]:
# Search some informations about limited 2 query from internet with max length 600

search_limit = 2
text_length_limit = 600
searched_data = {}

for e_query in pure_extracted_text[:search_limit]:
  query = e_query['arguments']['query']
  result = search_tool.invoke({
            "query": query,
            "num_results": 1
        })
  result = result.results[0].text
  final_text = extract_useful_text(result.lower(), query.lower())
  searched_data[query] = final_text[:text_length_limit]

print(searched_data)

In [ ]:
# Instructions for model to summarize the information from internet search

message_for_summarising = [

    {'role':'system',

     'content':""" you are good writer. I will give you json string. you need
     to do fact extraction, article generation and rewrite the input in
      your own writing style to make human readable article, so firstly you count
      how many parts include in the json.
     then you think how to rewrite paragraph for each topic with title.
      Example there are 3 different topics in string,  rewrite 3 paragraphs
       with each titles to make one topic.
      i want you to combine and rewrite these input in one useful article.
      make sure your answer must be in <answer> and </answer>"""},

    {'role': 'user',

     'content':json.dumps(searched_data)}

]

In [ ]:
# Making Chat Template for Information Summarising

text_for_summarising = tokenizer.apply_chat_template(
    message_for_summarising,
    tokenize=False,
)

In [ ]:
# Transforming Texts to Vectors and Generate

tokens_for_summarising = tokenizer(text_for_summarising,return_tensors="pt")

output_tokens_summarising = llm_model.generate(**tokens_for_summarising, max_new_tokens=800)

In [ ]:
# Taking only new generated texts

new_tokens_summarising = output_tokens_summarising[0][tokens_for_summarising['input_ids'].shape[1]:]

output_text_summarising = tokenizer.decode(new_tokens_summarising, skip_special_tokens=True)[0]

print(output_text_summarising)

In [ ]:
# Extract The Final Answer from Generated Texts

final_result = max(
    re.findall(
        r"<answer>\s*(.*?)\s*</answer>",
        output_text_summarising),
    key=len,
    default="Sorry I don't have any information."
    )

final_result = final_result.replace('\\n','\n')

print(final_result)